# BLOQUE 1_D5C3V1 – Infraestructura peatonal
## Objetivo
Construir capa consolidada ECONOMICA normalizada por unidad espacial.
## Inputs
- censo.csv
- manzana.shp

## Procesos
- Limpieza
- Homologación de CRS
- Join espacial
- Normalización

## Output
1_D5C3_Movilidad.gpkg
- 1_D5C3V1_Infraestructura_peatonal_mzn.shp
	
	

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
from dbfread import DBF
import jenkspy
import json
from jenkspy import jenks_breaks
import time

In [6]:
# Rutas
# LEER INFORMACIÓN CENSAL
Censo_Urbn_ruta = r"C:\Users\Alejandro\Desktop\DAL_LAB\data_raw\Censos\Censo2020_CEU_jal_csv\TI_MANZANA_EU_14.csv"
manzanas_ruta = r"C:\Users\Alejandro\Desktop\DAL_LAB\data_raw\Manzanas\14m.shp"

ruta_salida = r"C:\Users\Alejandro\Desktop\DAL_LAB\outputs\etapa_1\1_D5C3_Movilidad.gpkg"

In [7]:
# Cargar los GeoDataFrames
MZNA_2020_df = pd.read_csv(Censo_Urbn_ruta, dtype=str)
gdf_manzanas = gpd.read_file(manzanas_ruta)

## Tratado Censo Entorno

In [10]:
# Filtrar censos
MZNA_2020_df_F = MZNA_2020_df[['ENT', 'MUN', 'LOC', 'AGEB', 'MZA', 'ARBOLES_C', 'ALUMPUB_C', 'BANQUETA_C']].copy()

In [11]:
# Lista de columnas a procesar
campos = ['ARBOLES_C', 'ALUMPUB_C', 'BANQUETA_C']

# Diccionario de reemplazo
replacement_map = {
    "1": 100,
    "2": 40,
    "3": 0,
    "8": 0,
    "9": 0
}

for field in campos:
    if field in MZNA_2020_df_F.columns:
        MZNA_2020_df_F.loc[:, field] = (
            MZNA_2020_df_F[field]
            .replace(replacement_map)
            .where(lambda s: s.isin(replacement_map.values()))
        )

In [13]:
#Crear CVEGEO
MZNA_2020_df_F['CVEGEO'] = MZNA_2020_df_F['ENT'] + MZNA_2020_df_F['MUN'] + MZNA_2020_df_F['LOC'] + MZNA_2020_df_F['AGEB'] + MZNA_2020_df_F['MZA']

# Renombrar columnas de ITER_2020_df antes del merge para evitar conflictos
gdf_manzanas['CVEGEO'] = gdf_manzanas['CVEGEO'].astype(str)
MZNA_2020_df_F['CVEGEO'] = MZNA_2020_df_F['CVEGEO'].astype(str)

renombres = {
    "ARBOLES_C": "ARB", "ALUMPUB_C": "ALU", "BANQUETA_C": "BAN",
}
MZNA_2020_df_F = MZNA_2020_df_F.rename(columns=renombres)

# Obtener dinámicamente las columnas renombradas para cada DataFrame
columnas_mzna = ['CVEGEO', 'ARB', 'ALU', 'BAN']

# Realizar los merges
gdf_manzanas = gdf_manzanas.merge(
    MZNA_2020_df_F[columnas_mzna],
    on='CVEGEO', 
    how='left'
)

In [15]:
gdf_manzanas_fill = gdf_manzanas.fillna(0)

In [17]:
# Guardar el GeoPackage base actualizado
gdf_manzanas_fill.to_file(ruta_salida, layer='1_D5C3V1_Infraestructura_peatonal_mzn', driver="GPKG")

In [27]:
# Crear un buffer de 10 metros alrededor de las manzanas
gdf_manzanas_buffer = gdf_manzanas.copy()
gdf_manzanas_buffer['geometry'] = gdf_manzanas_buffer.geometry.buffer(10)

# Realizar la unión espacial entre las manzanas y las unidades económicas
gdf_union = gpd.sjoin(gdf_manzanas_buffer, gdf_denue, how="left", predicate="contains")

# Contar el número de unidades económicas dentro de cada manzana
numero_unidades_por_manzana = gdf_union.groupby('CVEGEO').size().reset_index(name='UE')

# Unir los resultados al GeoDataFrame de las manzanas
gdf_manzanas_final = gdf_manzanas.merge(numero_unidades_por_manzana, on='CVEGEO', how='left')

# Mostrar el resultado
gdf_manzanas_final

,CVEGEO,CVE_ENT,CVE_MUN,CVE_LOC,CVE_AGEB,CVE_MZA,AMBITO,TIPOMZA,geometry,UE
0,1400100010226003,14,001,0001,0226,003,Urbana,Típica,"POLYGON ((2406377.75 974961.594, 2406376.127 9...",4
1,1400100010211015,14,001,0001,0211,015,Urbana,Típica,"POLYGON ((2406681.686 975531.995, 2406674.665 ...",1
2,1400100010211011,14,001,0001,0211,011,Urbana,Típica,"POLYGON ((2406061.676 975289.259, 2406069.062 ...",13
3,1400100010207020,14,001,0001,0207,020,Urbana,Típica,"POLYGON ((2405836.806 974518.055, 2405844.876 ...",8
4,1400100010194015,14,001,0001,0194,015,Urbana,Típica,"POLYGON ((2405774.76 975251.892, 2405774.328 9...",5
...,...,...,...,...,...,...,...,...,...,...
140999,141250052012A040,14,125,0052,012A,040,Rural,Típica,"POLYGON ((2450269.737 963843.34, 2450268.438 9...",1
141000,141250052012A041,14,125,0052,012A,041,Rural,Típica,"POLYGON ((2450228.916 963905.119, 2450273.154 ...",1
141001,141250052012A042,14,125,0052,012A,042,Rural,Típica,"POLYGON ((2450273.461 963909.843, 2450228.419 ...",1
141002,1412500640115025,14,125,0064,0115,025,Rural,Típica,"POLYGON ((2445003.343 977547.067, 2444993.892 ...",1


In [28]:
# Guardar el GeoPackage con los nuevos campos
gdf_manzanas_final.to_file(ruta_salida, layer='1_D3C1V1_Mzn_Unidades_Economicas', driver='GPKG')